# SuGuard ML — training and honest evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/murzakhmetov/suguard/blob/main/ml/notebooks/SuGuard_ML_Colab.ipynb)

This notebook reproduces the training in `ml/` and adds the two checks that need public datasets
which could not be downloaded where the code was written: heart rate under **real** wrist motion
and the glucose hypothesis on **real** paired data.

| Step | What | Data | Time |
|---|---|---|---|
| 1 | Tests | — | 3 min |
| 2 | Diabetes-risk model | NHANES 2017–2018 (real, laboratory HbA1c) | 2 min |
| 3 | What fingersticks alone achieve | real CGM (iglu) | 1 min |
| 4 | Denoiser networks: results of the shipped models; optional retraining | simulated MAX30102 | 1 min (retraining: 1–1.5 h) |
| 5 | Heart rate under **real** motion | PPG-DaLiA (UCI) | 20–40 min |
| 6 | **Glucose**: test of the hypothesis on real data | BIG IDEAs (PhysioNet): wrist PPG + CGM + meals | 30–60 min |
| 7 | Download the results | — | 1 min |

**How to read the glucose result (step 6).** A model is useful only if, on *people it has never
seen* (protocol B) and on *later days* after calibration (protocol C), it beats the simple
baselines ("predict the mean", "last fingerstick") **and** the PPG features add accuracy beyond
context (time of day, meals). Otherwise the model file says `validated: false` and the API does
not present its output as a measurement. For BIG IDEAs `validated` is always false (another
sensor, 16 people): what matters there is the gain over the baselines and the controls.

Steps 1–3 and 5–7 run on a CPU. For retraining in step 4 choose `Runtime → Change runtime type → T4 GPU` first.

## 0. Setup

In [ ]:
import glob, json, os, sys, zipfile
if os.path.basename(os.getcwd()) != 'ml':
    if not os.path.exists('suguard'):
        # Private repository: replace the URL by https://<TOKEN>@github.com/murzakhmetov/suguard.git
        !git clone -q -b main https://github.com/murzakhmetov/suguard.git
    %cd suguard/ml
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 1. Tests
The package's own tests (76), then the firmware's signal core compiled for this machine and
compared with the Python reference window by window (needs `g++`; the RISC-V tests are skipped here).

In [ ]:
!python -m pytest -q
!python -m pytest -q ../firmware/tests

## 2. Diabetes-risk model (NHANES)
Trained on 4,318 adults without a diabetes diagnosis; the labels are laboratory HbA1c (≥ 5.7 %:
prediabetes or diabetes, ≥ 6.5 %: diabetes). Every metric is out-of-fold (5 × 5-fold
cross-validation), i.e. measured on people the model did not see.

In [ ]:
!python scripts/train_risk.py --out models
r = json.load(open('models/metrics/risk_cv.json'))
for v, d in r['variants'].items():
    print(f"{v:20s}", {t: f"AUC {m['auc']:.3f}" for t, m in d.items()})
print(json.dumps(r['categories_oof'], indent=1))
for t, g in r['subgroup_calibration'].items():
    for k, v in g.items():
        print(f"{t:12s} {k:45s} predicted {v['mean_predicted']:.3f}  observed {v['observed']:.3f}")

Optional: the survey-weighted variant (representative of the US population). It needs the CDC
demographics file and writes to a separate folder, so the shipped model is not replaced.

In [ ]:
!wget -q https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.xpt -O DEMO_J.xpt && python scripts/train_risk.py --out models_weighted --weights DEMO_J.xpt | tail -3

## 3. What fingersticks alone achieve (real CGM)
The bar for any non-invasive model: how well glucose is "predicted" from the latest calibration
fingersticks alone, without a bracelet.

In [ ]:
!python scripts/benchmark_calibration.py --out models/metrics
c = json.load(open('models/metrics/calibration_baselines_cgm.json'))
for cohort, d in c.items():
    for sched in ('1_per_day', '2_per_day', '4_per_day'):
        print(f"{cohort:6s} {sched:10s}", {m: f"MARD {v['mard_pct']:.1f} %" for m, v in d[sched].items() if isinstance(v, dict)})

## 4. Denoiser networks
The repository contains both trained networks (server: `models/denoiser.onnx`; bracelet:
`models/denoiser_edge_int.npz`, compiled into the firmware). First their measured results:

In [ ]:
srv = json.load(open('models/metrics/denoiser_eval.json'))
edge = json.load(open('models/metrics/denoiser_edge_eval.json'))
print('Whole pipeline with every reliability check, 600 simulated windows:')
for name, r in (('classical', srv['pipeline_with_gates']['classical']), ('bracelet', edge['pipeline_with_gates']['denoiser']),
                ('server', srv['pipeline_with_gates']['denoiser'])):
    print(f"  {name:10s} heart rate in {100 * r['hr_coverage']:3.0f} % of windows, MAE {r['hr_mae']:.2f} bpm, errors > 10 bpm: {r['hr_errors_over_10bpm']}"
          f" | SpO2 in {100 * r['spo2_coverage']:3.0f} %, MAE {r['spo2_mae']:.2f}, errors > 4 points: {r['spo2_errors_over_4pts']}")
print('Windows without motion, by pulse size (heart rate / SpO2 given; "rule off": without the low-perfusion rule):')
for b in srv['rest_by_perfusion']['classical']:
    print(f"  {b:11s}", {name: "{:.0f} % / {:.0f} %".format(100 * src['rest_by_perfusion'][key][b]['hr_coverage'],
                                                         100 * src['rest_by_perfusion'][key][b]['spo2_coverage'])
                         for name, src, key in (('classical', srv, 'classical'), ('bracelet', edge, 'denoiser'),
                                                ('bracelet, rule off', edge, 'denoiser_without_low_perfusion_fallback'),
                                                ('server', srv, 'denoiser'),
                                                ('server, rule off', srv, 'denoiser_without_low_perfusion_fallback'))})
print('Inputs that must not produce a value (heart rate reported / SpO2 reported / windows):')
for name, m in (('classical', srv['must_not_report']['classical']), ('bracelet', edge['must_not_report']['denoiser']),
                ('server', srv['must_not_report']['denoiser'])):
    print(f"  {name:10s}", {k: f"{v['heart_rate_reported']}/{v['spo2_reported']}/{v['windows']}" for k, v in m.items()})
print('Clean windows of pulse shapes generated outside the training simulator (share that gets a heart rate):')
for shape in srv['shape_robustness']:
    print(f"  {shape:36s}", {name: f"{100 * src['shape_robustness'][shape][key]['hr_coverage']:.0f} %"
                              for name, src, key in (('classical', srv, 'classical'), ('bracelet', edge, 'denoiser'),
                                                     ('server', srv, 'denoiser'))})
print('Known hard cases (heart rate reported / wrong, SpO2 reported / wrong; see the README):')
for case in srv['hard_cases']['classical']:
    print(f"  {case:46s}", {name: "{heart_rate_reported}/{heart_rate_wrong}, {spo2_reported}/{spo2_wrong}".format(**src['hard_cases'][key][case])
                             for name, src, key in (('classical', srv, 'classical'), ('bracelet', edge, 'denoiser'),
                                                    ('server', srv, 'denoiser'))})

**Optional: retrain.** Set `RETRAIN = True` to regenerate the 200,000 training windows and train
both networks from scratch with the settings of the shipped models (about 1–1.5 h with a T4 GPU,
most of it generating the windows on Colab's two CPUs). The last command checks that the firmware
reproduces the new bracelet model bit for bit. If you keep a retrained model, commit **together**
`ml/models/*`, `ml/models/metrics/*` and `firmware/suguard/src/core/sg_model.h` (step 7 puts them
in one archive).

In [ ]:
RETRAIN = False
if RETRAIN:
    !python scripts/make_denoiser_dataset.py --out data/denoiser --train 200000 --val 4000 --test 4000 --workers 2
    # server model
    !python scripts/train_denoiser.py --data data/denoiser --steps 16000 --bs 128 --out models
    !python scripts/eval_denoiser.py --data data/denoiser --out models/metrics
    # bracelet model: train, convert to integers, write the firmware header, evaluate the integer model
    !python scripts/train_denoiser.py --data data/denoiser --arch edge --steps 16000 --bs 128 --out data/edge_s
    !python scripts/export_edge_model.py --ckpt data/edge_s/denoiser_edge.pt --data data/denoiser
    !python scripts/eval_denoiser.py --data data/denoiser --edge-int models/denoiser_edge_int.npz --name denoiser_edge_eval --out models/metrics
    !python -m pytest -q ../firmware/tests

## 5. Heart rate under real motion: PPG-DaLiA
15 people, about 2.5 h each of everyday activity with a wrist PPG sensor, reference heart rate
from ECG. This is the check the simulation cannot give: **when the pipeline shows a heart rate
during real motion, is it right, and how often does it show one?**

The Empatica E4 records one green channel in arbitrary units, so the script has to assume a light
level and feeds the same signal as "red" and "infrared": the checks that compare the two
wavelengths pass trivially and the network cannot use the red/IR motion cue. Read the result as a
lower bound on what the two-wavelength bracelet can reject.

In [ ]:
if not glob.glob('**/S1/S1.pkl', recursive=True):
    !wget -q -O dalia.zip "https://archive.ics.uci.edu/static/public/495/ppg+dalia.zip"
    zipfile.ZipFile('dalia.zip').extractall('dalia')
    for z in glob.glob('dalia/**/*.zip', recursive=True):      # the archive contains another archive
        zipfile.ZipFile(z).extractall('dalia')
dalia_root = os.path.dirname(os.path.dirname(glob.glob('**/S1/S1.pkl', recursive=True)[0]))
!python scripts/eval_dalia.py --root "$dalia_root" --out models/metrics

In [ ]:
d = json.load(open('models/metrics/dalia_hr_eval.json'))
g = d['gated_pipeline']
print('which way up the E4 signal was taken (+1 as recorded), per subject:', g['polarity_by_subject'])
paths = [p for p in ('classical', 'bracelet', 'server') if p in g]
for p in paths:
    r = g[p]
    if r['hr_mae'] is None:
        print(f"{p:10s} no heart rate shown in any of {r['windows']} windows"); continue
    print(f"{p:10s} heart rate shown in {100 * r['hr_coverage']:.0f} % of {r['windows']} windows; MAE {r['hr_mae']:.2f} bpm, "
          f"95th percentile {r['hr_p95']:.1f}, errors > 10 bpm: {r['hr_errors_over_10bpm']} "
          f"({100 * r['share_of_given_values_over_10bpm']:.1f} % of the shown values)")
print()
print(f"{'activity':14s}" + ''.join(f"{p + ': shown / MAE':>26s}" for p in paths))
for act in g[paths[0]]['by_activity']:
    cells = []
    for p in paths:
        a = g[p]['by_activity'][act]
        cells.append(f"{100 * a['hr_coverage']:5.0f} % / " + (f"{a['hr_mae']:5.1f} bpm" if a['hr_mae'] is not None else '    -    '))
    print(f"{act:14s}" + ''.join(f"{c:>26s}" for c in cells))
print()
print('Every window, no checks (spectral tracking), mean error in bpm:', {k: round(v, 1) for k, v in d['spectral_tracking']['mean_mae'].items()})
# What to look for: the share of shown values that are off by more than 10 bpm. In simulation it is
# 0 % with the networks. If it is clearly above that here, the reliability checks need real-motion
# data from the bracelet before the numbers can be trusted during movement.

## 6. Glucose: test of the hypothesis on real data (BIG IDEAs)
16 adults (normal to prediabetic), Empatica E4 wristband (PPG at 64 Hz) + Dexcom G6 CGM for 8–10
days + a food log. The download is a few GB from the AWS Open Data mirror (no account needed).

In [ ]:
!pip install -q awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/big-ideas-glycemic-wearable/1.1.2/ big-ideas
# fallback (slower):
# !wget -r -N -c -np -nH --cut-dirs=3 -P big-ideas https://physionet.org/files/big-ideas-glycemic-wearable/1.1.2/
!ls big-ideas | head

In [ ]:
!python scripts/build_bigideas_table.py --root big-ideas --out data/bigideas_paired.parquet --every-n 3
!python scripts/train_glucose.py --table data/bigideas_paired.parquet --out models/bigideas --name bigideas

In [ ]:
rep = json.load(open('models/bigideas/metrics/bigideas_eval.json'))
def row(name, m):
    if not m:
        print(f"{name:52s} could not be computed"); return
    r2 = 'n/a' if m['r2'] is None else f"{m['r2']:.2f}"
    print(f"{name:52s} MARD {m['mard_pct']:5.1f} %  MAE {m['mae_mgdl']:5.1f} mg/dL  R2 {r2:>5s}  Clarke A+B {m['clarke_ab_pct']:5.1f} %")
row('A: random split (LEAKY, inflated)', rep.get('A_random_rows_LEAKY'))
b = rep.get('B_subject_wise') or {}
row('B: new people - model', b.get('model'))
row('B: new people - "predict the mean"', b.get('baseline_train_mean'))
c = rep.get('C_personalised') or {}
if c:
    print(f"C: {c.get('n_subjects')} people scored, not scored: {c.get('subjects_not_scored')}")
    row('C: model + calibration', c.get('model_plus_calibration'))
    for k in ('calibration_only_kalman', 'last_fingerstick_carried_forward', 'personal_time_of_day_profile'):
        row('C: ' + k, c.get(k))
for k, v in (rep.get('controls') or {}).items():
    row(f'control {k} (B)', (v or {}).get('B')); row(f'control {k} (C)', (v or {}).get('C'))
print('relative MARD gain:', json.dumps(rep.get('relative_mard_gain'), indent=1))
print('validated =', rep.get('validated'))
print('reasons:', *rep.get('not_validated_because', []), sep='\n  - ')
# 'validated' is always False for BIG IDEAs: it is not SuGuard data, and 16 people are too few.
# Look at the gain over the baselines and over the two controls: it tells whether wrist PPG
# carries information about glucose at all. A gain around zero means it does not show here.

## 7. Download the results
Download the archive and send it (or just `models/metrics/*.json`): the numbers of steps 5 and 6
decide what the next step is.

In [ ]:
!zip -q -r suguard_ml_results.zip models ../firmware/suguard/src/core/sg_model.h
from google.colab import files
files.download('suguard_ml_results.zip')